# SFT v3: yangi data, Qwen2.5-1.5B, LoRA va QLoRA (Kaggle GPU)

v2 bilan bir xil model va sozlamalar, faqat **data yangi** (`sft_v3/data/`): v2 -> v3 farqi datadan keladi.
Yagona boshqa o'zgarish: 5 emas, 3 epoch (v2 da ikkala usul ham 2-epochdan keyin overfit bo'lgan).

Boshlashdan oldin o'ng tomondagi **Settings** bo'limida:

- **Accelerator -> GPU T4 x2** (yoki GPU P100)
- **Internet -> On**

Keyin **Run All** ni bosing.

## 1. GPU borligini tekshirish

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Loyiha va base modelni yuklab olish

In [ ]:
%cd /kaggle/working
!rm -rf Post_Training
!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/davronbek-malikov/Post_Training.git
%cd Post_Training
!pip -q install -U transformers peft bitsandbytes accelerate
!pip -q uninstall -y torchao  # Kaggle'dagi eski torchao peft bilan adapter yuklashni buzadi

from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-1.5B", local_dir="Qwen2.5-1.5B")
!ls -lh Qwen2.5-1.5B/*.safetensors
!wc -l sft_v3/data/train.jsonl sft_v3/data/val.jsonl

## 3. LoRA bilan o'qitish (v3 data)

In [ ]:
!python sft_v2/train.py --method lora --data sft_v3/data --name sft-lora-v3 --epochs 3 --rank 64

In [ ]:
!rm -f /kaggle/working/sft_v3_adapters.zip
!zip -q -r /kaggle/working/sft_v3_adapters.zip chat_ui/adapters/sft-lora-v3
!ls -lh /kaggle/working/sft_v3_adapters.zip

## 4. QLoRA bilan o'qitish (v3 data)

Chiqishning boshida `QLoRA: ... ta qatlam 4-bit` degan qator bo'lishi kerak.

In [ ]:
!python sft_v2/train.py --method qlora --data sft_v3/data --name sft-qlora-v3 --epochs 3 --rank 64

In [ ]:
!zip -q -r /kaggle/working/sft_v3_adapters.zip chat_ui/adapters/sft-qlora-v3
!ls -lh /kaggle/working/sft_v3_adapters.zip

## 5. Baholash: 68 ta test savoli

v2 (1.5B, eski data): LoRA 44/68, QLoRA 42/68.

In [ ]:
!python eval/run_eval.py --base Qwen2.5-1.5B --model sft-lora-v3 | tail -12
!python eval/run_eval.py --base Qwen2.5-1.5B --model sft-qlora-v3 | tail -12
!zip -q /kaggle/working/sft_v3_adapters.zip eval/results/sft-lora-v3.json eval/results/sft-qlora-v3.json

## 6. Yuklab olish

`sft_v3_adapters.zip` ni yuklab oling va loyiha papkasida oching (yoki `sft_v3/` ichiga qo'ying).

In [ ]:
%cd /kaggle/working
from IPython.display import FileLink
FileLink("sft_v3_adapters.zip")